In [1]:
%load_ext autoreload 
%autoreload 2

Alter data prep to be exactly like the Deep Triangle paper. Each Dev period will have a target SEQUENCE and an input SEQUENCE.
<br><br> Later down the line we can choose it to be either an Autoregressive MODEL or a SEQ2SEQ model. 


In [2]:

from rnn_reserving.data_import import read_local_raw_data, process_data, split_data
import pandas as pd


In [3]:
df_cas = read_local_raw_data()
    
df_cas = process_data(df_cas)
df_cas = split_data(df_cas)

In [4]:
df_test = df_cas[df_cas['GRCODE'] == 43][
    ['AccidentYear',
    'DevelopmentLag',
    'incurred_loss_ratio',
    'paid_loss_ratio',
    'case_loss_ratio',
    'calendar_year',
    'GRCODE_mapped',
    'bucket'
    ]
].copy()

In [5]:
triangle = (
    df_test
    .pivot_table(
        index="AccidentYear",
        columns="DevelopmentLag",
        values="paid_loss_ratio",
        aggfunc="sum" 
    )
    .sort_index()
)
triangle

DevelopmentLag,1,2,3,4,5,6,7,8,9,10
AccidentYear,,,,,,,,,,
1988,0.148603,0.372067,0.481564,0.636872,0.687151,0.687151,0.687151,0.686034,0.686034,0.686034
1989,0.274141,0.512474,0.694159,0.756971,0.810977,0.870561,0.862929,0.874083,0.874083,0.862342
1990,0.344710,0.825947,1.168280,1.373238,1.459501,1.484632,1.488029,1.487859,1.482255,1.494651
1991,0.270317,0.686785,0.901037,0.992374,1.031995,1.076978,1.090801,1.108317,1.098308,1.100095
1992,0.273592,0.580931,0.812566,0.911636,0.959128,0.981083,0.983506,1.001194,0.999928,1.000543
1993,0.237435,0.566338,0.760772,0.856025,0.889322,0.908048,0.910382,0.923530,0.926063,0.925807
1994,0.294918,0.631898,0.794907,0.899847,0.941509,0.974534,0.986527,0.988241,0.990189,0.990846
1995,0.282118,0.546138,0.665078,0.730542,0.773499,0.801088,0.814126,0.815680,0.820676,0.821768
1996,0.268576,0.499606,0.602549,0.672335,0.715954,0.739180,0.750836,0.752094,0.752520,0.752797


The below shows train in blue, val in green and test in red! Yay! 


In [6]:
ilr_triangle = (
    df_test
    .pivot(
        index="AccidentYear",
        columns="DevelopmentLag",
        values="paid_loss_ratio"
    )
    .sort_index()
)

bucket_triangle = (
    df_test
    .pivot(
        index="AccidentYear",
        columns="DevelopmentLag",
        values="bucket"
    )
    .sort_index()
)

def style_bucket(val):
    if val == "train":
        return "background-color: #cce5ff; color: #003366;"
    if val == "validation":
        return "background-color: #d4edda; color: #155724;"
    if val == "test":
        return "background-color: #f8d7da; color: #721c24;"
    return ""


styled = (
    ilr_triangle
    .style
    .apply(
        lambda _: bucket_triangle.applymap(style_bucket),
        axis=None
    )
)

styled


C:\Users\TobyCook\AppData\Local\Temp\ipykernel_11344\2490517049.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lambda _: bucket_triangle.applymap(style_bucket),


DevelopmentLag,1,2,3,4,5,6,7,8,9,10
AccidentYear,,,,,,,,,,
1988,0.148603,0.372067,0.481564,0.636872,0.687151,0.687151,0.687151,0.686034,0.686034,0.686034
1989,0.274141,0.512474,0.694159,0.756971,0.810977,0.870561,0.862929,0.874083,0.874083,0.862342
1990,0.344710,0.825947,1.168280,1.373238,1.459501,1.484632,1.488029,1.487859,1.482255,1.494651
1991,0.270317,0.686785,0.901037,0.992374,1.031995,1.076978,1.090801,1.108317,1.098308,1.100095
1992,0.273592,0.580931,0.812566,0.911636,0.959128,0.981083,0.983506,1.001194,0.999928,1.000543
1993,0.237435,0.566338,0.760772,0.856025,0.889322,0.908048,0.910382,0.923530,0.926063,0.925807
1994,0.294918,0.631898,0.794907,0.899847,0.941509,0.974534,0.986527,0.988241,0.990189,0.990846
1995,0.282118,0.546138,0.665078,0.730542,0.773499,0.801088,0.814126,0.815680,0.820676,0.821768
1996,0.268576,0.499606,0.602549,0.672335,0.715954,0.739180,0.750836,0.752094,0.752520,0.752797


In [7]:
# We do need to handle the val data having a full history for input but only a partial for target though..

In [ ]:
from collections import defaultdict

def build_sequences(df):
    """Build sequences for RNN training, validation, and testing."""
    feature_cols = ['paid_loss_ratio']

    data = {
        'train': defaultdict(list),
        'validation': defaultdict(list),
        'test': defaultdict(list),
    }

    # Group once
    for (ay, cc), group in df.groupby(['AccidentYear', 'GRCODE_mapped']):
        group = group.sort_values('DevelopmentLag')

        dev_lags = group['DevelopmentLag'].values
        splits = group['bucket'].values
        
        val_counter = 0 # to track position in validation targets
        test_counter = 0 # to track position in test targets

        # iterate over valid sequence endpoints
        for i in range(1, len(group)):
            # get the bucket - train, test, or validation
            split = splits[i]

            if split == 'train':
                # then we just take the train ones (we don't want to train on val!)
                train_data = group[group['bucket'] == 'train']
                input_seq = train_data[feature_cols].values[:i]
                target_seq = train_data[feature_cols].values[i+1:, 0]

            elif split == 'validation':
                # we get the full history from train + val for input
                # but only val for target
                val_data = group[group['bucket'].isin(['train', 'validation'])]
                input_seq = val_data[feature_cols].values[:i]
                # We only want targets from the validation period (avoids leakage)
                val_data = group[group['bucket'] == 'validation']
                target_seq = val_data[feature_cols].values[val_counter:, 0]
                val_counter += 1
                
            elif split == 'test':
                # we get the full history from train + val + test for input
                # but only test for target
                test_data = group[group['bucket'].isin(['train', 'validation', 'test'])]
                input_seq = test_data[feature_cols].values[:i]
                # We only want targets from the test period (avoids leakage)
                test_data = group[group['bucket'] == 'test']
                target_seq = test_data[feature_cols].values[test_counter:, 0]
                test_counter += 1
            else:
                raise ValueError(f"Unknown bucket: {split}")

            data[split]['inputs'].append(input_seq)
            data[split]['targets'].append(target_seq)
            data[split]['lengths'].append(len(input_seq))
            data[split]['ids'].append((ay, cc, dev_lags[i], split))

    return data

data = build_sequences(df_test)

data['test']['inputs'][3], data['test']['targets'][2]


def pad_sequences(sequences, pad_value=0.0):
    """Pad sequences to the same length."""
    # pad these badd boiss..
    # ok we must go to the 10 and post pad inputs... and post pad the targets to max length... Actually not htat hard, but really must go to Lidl now... 

    ... 

i: 1 split: train
input_seq: [[0.14860335]] target_seq: [0.48156425 0.63687151 0.68715084 0.68715084 0.68715084 0.68603352]
i: 2 split: train
input_seq: [[0.14860335]
 [0.37206704]] target_seq: [0.63687151 0.68715084 0.68715084 0.68715084 0.68603352]
i: 3 split: train
input_seq: [[0.14860335]
 [0.37206704]
 [0.48156425]] target_seq: [0.68715084 0.68715084 0.68715084 0.68603352]
i: 4 split: train
input_seq: [[0.14860335]
 [0.37206704]
 [0.48156425]
 [0.63687151]] target_seq: [0.68715084 0.68715084 0.68603352]
i: 5 split: train
input_seq: [[0.14860335]
 [0.37206704]
 [0.48156425]
 [0.63687151]
 [0.68715084]] target_seq: [0.68715084 0.68603352]
i: 6 split: train
input_seq: [[0.14860335]
 [0.37206704]
 [0.48156425]
 [0.63687151]
 [0.68715084]
 [0.68715084]] target_seq: [0.68603352]
i: 7 split: train
input_seq: [[0.14860335]
 [0.37206704]
 [0.48156425]
 [0.63687151]
 [0.68715084]
 [0.68715084]
 [0.68715084]] target_seq: []
i: 8 split: validation
input_seq: [[0.14860335]
 [0.37206704]
 [0.48

(array([[0.27031697],
        [0.68678503],
        [0.9010367 ],
        [0.99237369],
        [1.03199476],
        [1.07697807],
        [1.09080076]]),
 array([1.49465104]))

In [ ]:
from collections import defaultdict
## FROM CHATGPT: seems to want me to do a fixed time horizon approach?

##  IGNORE THIS - it's Chatbot GPT code.. But it doesn't work because it misses some validation sequences. 
def build_sequences(df, feature_cols):
    data = {
        "train": defaultdict(list),
        "validation": defaultdict(list),
        "test": defaultdict(list),
    }

    for (ay, cc), group in df.groupby(["AccidentYear", "GRCODE_mapped"]):

        print('Processing AY:', ay, 'GRCODE_mapped:', cc  )
        group = group.sort_values("DevelopmentLag").reset_index(drop=True)
        print(group)
        for i in range(len(group)):
            print(i)
            split = group.loc[i, "bucket"]
            print(split)
            # input = history up to valuation date i
            input_seq = group.loc[:i, feature_cols].values
            print('input_seq:', input_seq)
            # target = future values only, bucket-pure
            future_mask = (
                (group.index >= i) &
                (group["bucket"] == split)
            )

            target_seq = group.loc[future_mask, feature_cols[0]].values
            print('target_seq:', target_seq)
            # skip if no future targets (common near the end)
            if len(target_seq) == 0:
                continue

            data[split]["inputs"].append(input_seq)
            data[split]["targets"].append(target_seq)
            data[split]["lengths"].append(len(input_seq))
            data[split]["ids"].append(
                (ay, cc, group.loc[i, "DevelopmentLag"], split)
            )
        break 

    return data

data = build_sequences(df_test, feature_cols)
data['validation']['inputs'][0], data['validation']['targets'][0]

Processing AY: 1988 GRCODE_mapped: 0
   AccidentYear  DevelopmentLag  incurred_loss_ratio  paid_loss_ratio  \
0          1988               1             0.678212         0.148603   
1          1988               2             0.722905         0.372067   
2          1988               3             0.650279         0.481564   
3          1988               4             0.668156         0.636872   
4          1988               5             0.686034         0.687151   
5          1988               6             0.687151         0.687151   
6          1988               7             0.687151         0.687151   
7          1988               8             0.686034         0.686034   
8          1988               9             0.686034         0.686034   
9          1988              10             0.686034         0.686034   

   case_loss_ratio  calendar_year  GRCODE_mapped      bucket  
0         0.529609           1988              0       train  
1         0.350838           1989

(array([[0.14860335],
        [0.37206704],
        [0.48156425],
        [0.63687151],
        [0.68715084],
        [0.68715084],
        [0.68715084],
        [0.68603352],
        [0.68603352]]),
 array([0.68603352, 0.68603352]))

In [16]:
styled = (
    ilr_triangle
    .style
    .apply(
        lambda _: bucket_triangle.applymap(style_bucket),
        axis=None
    )
)

styled

C:\Users\TobyCook\AppData\Local\Temp\ipykernel_11344\1619337625.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lambda _: bucket_triangle.applymap(style_bucket),


DevelopmentLag,1,2,3,4,5,6,7,8,9,10
AccidentYear,,,,,,,,,,
1988,0.148603,0.372067,0.481564,0.636872,0.687151,0.687151,0.687151,0.686034,0.686034,0.686034
1989,0.274141,0.512474,0.694159,0.756971,0.810977,0.870561,0.862929,0.874083,0.874083,0.862342
1990,0.344710,0.825947,1.168280,1.373238,1.459501,1.484632,1.488029,1.487859,1.482255,1.494651
1991,0.270317,0.686785,0.901037,0.992374,1.031995,1.076978,1.090801,1.108317,1.098308,1.100095
1992,0.273592,0.580931,0.812566,0.911636,0.959128,0.981083,0.983506,1.001194,0.999928,1.000543
1993,0.237435,0.566338,0.760772,0.856025,0.889322,0.908048,0.910382,0.923530,0.926063,0.925807
1994,0.294918,0.631898,0.794907,0.899847,0.941509,0.974534,0.986527,0.988241,0.990189,0.990846
1995,0.282118,0.546138,0.665078,0.730542,0.773499,0.801088,0.814126,0.815680,0.820676,0.821768
1996,0.268576,0.499606,0.602549,0.672335,0.715954,0.739180,0.750836,0.752094,0.752520,0.752797


In [19]:
len(data['validation']['inputs'])

9

In [17]:
data['validation']['inputs'][0], data['validation']['targets'][0] , data['validation']['ids'][0]

(array([[0.14860335],
        [0.37206704],
        [0.48156425],
        [0.63687151],
        [0.68715084],
        [0.68715084],
        [0.68715084],
        [0.68603352],
        [0.68603352]]),
 array([0.68603352]),
 (np.int64(1988), np.int64(0), np.int64(9), 'validation'))

In [11]:
data['train']['inputs'][1], data['train']['targets'][1] , data['train']['ids'][1]

(array([[0.14860335],
        [0.37206704]]),
 array([0.48156425, 0.63687151, 0.68715084, 0.68715084, 0.68715084,
        0.68603352]),
 (np.int64(1988), np.int64(0), np.int64(2), 'train'))

In [12]:
data['validation']['inputs'][2], data['validation']['targets'][2] , data['validation']['ids'][2]

(array([[0.34471048],
        [0.82594668],
        [1.16827984],
        [1.37323824],
        [1.45950076],
        [1.48463237],
        [1.48802853]]),
 array([1.48785872]),
 (np.int64(1990), np.int64(0), np.int64(7), 'validation'))

In [13]:
data['train']['inputs'][10], data['train']['targets'][10], data['train']['ids'][10]  

(array([[0.27414147],
        [0.51247432],
        [0.69415908],
        [0.75697094]]),
 array([0.8109774 , 0.87056061, 0.86292926]),
 (np.int64(1989), np.int64(0), np.int64(4), 'train'))